# isitfair hardening: `bank`

Step-by-step audit of the `bank` prediction table built by
`hardening/datasets/registry.py` (see `hardening/datasets/CARDS.md`). The
audited model is the logistic-regression column `y_proba_lr`; every stage is
its own cell, timed with `perf_counter`. Warnings and exceptions are recorded
per stage in `RECORD` and written to `hardening/notebooks/results/`.

Environment knobs: `HARDEN_B` (bootstrap replicates, default 50) and
`HARDEN_STRICT` (`1` makes the last cell fail on any unexpected error).

In [1]:
import faulthandler
import json
import os
import sys
import time
import traceback
import warnings
from contextlib import contextmanager
from pathlib import Path

import numpy as np
import pandas as pd

for _c in (Path.cwd(), *Path.cwd().parents):
    if (_c / "hardening" / "datasets" / "registry.py").exists():
        REPO = _c
        break
sys.path.insert(0, str(REPO / "hardening" / "datasets"))
# Inside a Jupyter kernel sys.stderr has no file descriptor; give the watchdog a real file.
_FH = open(REPO / "hardening" / "logs" / "faulthandler_bank.log", "w")
faulthandler.dump_traceback_later(600, repeat=True, file=_FH)
import registry  # noqa: E402

import isitfair  # noqa: E402
from isitfair import FairnessAudit, GroupRecalibration, WassersteinPostprocessing, report_to_pdf  # noqa: E402

NAME = "bank"
B = int(os.environ.get("HARDEN_B", "50"))
STRICT = os.environ.get("HARDEN_STRICT", "0") == "1"
# Bootstrap workers for discrimination/calibration; results do not depend on it.
NJOBS = int(os.environ["HARDEN_NJOBS"]) if os.environ.get("HARDEN_NJOBS") else None
SEED = 42
OUT = REPO / "hardening" / "notebooks" / "results"
REPORTS = REPO / "hardening" / "reports"
OUT.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)
META = registry.meta(NAME)
THRESHOLD = META["threshold"]
RECORD = {"dataset": NAME, "B": B, "isitfair": isitfair.__version__,
          "isitfair_file": isitfair.__file__, "n_jobs": NJOBS, "stages": {}, "variants": {}}
print(f"isitfair {isitfair.__version__} from {isitfair.__file__}")
print(f"{NAME}: B={B} threshold={THRESHOLD} strict={STRICT} n_jobs={NJOBS}")


@contextmanager
def stage(name, expect_error=False):
    """Time a stage; record its warnings and any exception, then carry on."""
    entry = {"seconds": None, "warnings": [], "error": None, "expected_error": expect_error}
    RECORD["stages"][name] = entry
    t0 = time.perf_counter()
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        try:
            yield entry
        except Exception as exc:  # recorded, not raised, so later stages still run
            entry["error"] = f"{type(exc).__name__}: {exc}"
            entry["traceback"] = traceback.format_exc()
            print(f"!! {name} raised {entry['error']}")
    entry["seconds"] = round(time.perf_counter() - t0, 3)
    seen = {}
    for w in caught:
        key = f"{w.category.__name__}: {str(w.message)[:300]}"
        seen[key] = seen.get(key, 0) + 1
    entry["warnings"] = [f"{k} (x{v})" for k, v in seen.items()]
    print(f"[{name}] {entry['seconds']:.2f} s, {len(caught)} warning(s)"
          + (f", ERROR {entry['error']}" if entry["error"] else ""))
    for k in entry["warnings"][:8]:
        print("   ", k)
    (OUT / f"{NAME}_B{B}.json").write_text(json.dumps(RECORD, indent=2, default=str))

isitfair 0.4.0 from C:\Users\hugoa\Desktop\isitfair\src\isitfair\__init__.py
bank: B=50 threshold=0.2 strict=True n_jobs=-1


## 1. Load and validate inputs

In [2]:
with stage("load_validate"):
    df = registry.load(NAME)
    ATTRS = registry.ATTRS[NAME]
    checks = {
        "n": len(df),
        "index_is_range": isinstance(df.index, pd.RangeIndex),
        "y_true_dtype": str(df["y_true"].dtype),
        "y_true_values": sorted(df["y_true"].dropna().unique().tolist()),
        "y_true_missing": int(df["y_true"].isna().sum()),
        "y_proba_range": [float(df["y_proba_lr"].min()), float(df["y_proba_lr"].max())],
        "y_proba_missing": int(df["y_proba_lr"].isna().sum()),
        "attr_dtypes": {a: str(df[a].dtype) for a in ATTRS},
        "attr_missing": {a: int(df[a].isna().sum()) for a in ATTRS},
        "attr_levels": {a: int(df[a].nunique()) for a in ATTRS},
    }
    RECORD["checks"] = checks
    print(json.dumps(checks, indent=1))

{
 "n": 13564,
 "index_is_range": true,
 "y_true_dtype": "int64",
 "y_true_values": [
  0,
  1
 ],
 "y_true_missing": 0,
 "y_proba_range": [
  0.0005722501559348,
  0.9362851810567004
 ],
 "y_proba_missing": 0,
 "attr_dtypes": {
  "age_group": "object",
  "job": "object",
  "marital": "object"
 },
 "attr_missing": {
  "age_group": 0,
  "job": 0,
  "marital": 0
 },
 "attr_levels": {
  "age_group": 5,
  "job": 12,
  "marital": 3
 }
}
[load_validate] 0.02 s, 0 warning(s)


A missing subgroup label must be rejected with an actionable message. If any
attribute has missing labels, try the raw column first (recorded), then
recode to an explicit `"Unknown"` category for the rest of the notebook.

In [3]:
missing_attrs = [a for a in ATTRS if df[a].isna().any()]
if missing_attrs:
    with stage("raw_missing_labels", expect_error=True):
        FairnessAudit(y_true=df["y_true"], y_proba=df["y_proba_lr"],
                      sensitive_features={a: df[a] for a in ATTRS},
                      threshold=THRESHOLD, n_bootstrap=B, random_state=SEED, n_jobs=NJOBS)
    for a in missing_attrs:
        df[a] = df[a].fillna("Unknown")
print("recoded to 'Unknown':", missing_attrs or "none")

recoded to 'Unknown': none


In [4]:
y = df["y_true"].to_numpy(dtype=float)
p = df["y_proba_lr"].to_numpy(dtype=float)
marginals = {a: df[a].to_numpy(dtype=object) for a in ATTRS}
INTERSECTION = tuple(ATTRS[:2]) if len(ATTRS) >= 2 else None
sf = dict(marginals)
if INTERSECTION:
    sf[INTERSECTION] = None
print("intersection:", INTERSECTION)

intersection: ('age_group', 'job')


## 2. Construct the audit

In [5]:
with stage("construct"):
    audit = FairnessAudit(y_true=y, y_proba=p, sensitive_features=sf, threshold=THRESHOLD,
                          n_bootstrap=B, random_state=SEED, n_jobs=NJOBS)
    print(audit)

FairnessAudit(n=13564, events=1587, threshold=0.2, n_bootstrap=50, random_state=42, attributes: age_group (5), job (12), marital (3), age_group × job (58))
[construct] 0.06 s, 0 warning(s)


## 3. Discrimination

In [6]:
with stage("discrimination"):
    disc = audit.discrimination()
    print(disc[disc.subgroup != "GAP"][["attribute", "subgroup", "n", "n_events", "auroc",
                                         "auroc_ci_low", "auroc_ci_high"]].to_string(index=False))

      attribute              subgroup       n  n_events    auroc  auroc_ci_low  auroc_ci_high
      age_group               Overall 13564.0    1587.0 0.751137      0.736797       0.765831
      age_group                 18-29  1503.0     255.0 0.754352      0.727009       0.797985
      age_group                 30-39  5463.0     577.0 0.727907      0.705149       0.754646
      age_group                 40-49  3457.0     302.0 0.717095      0.678549       0.752744
      age_group                 50-59  2615.0     268.0 0.749536      0.705722       0.780564
      age_group                   60+   526.0     185.0 0.720377      0.675158       0.767371
            job               Overall 13564.0    1587.0 0.751137      0.736797       0.765831
            job                admin.  1509.0     179.0 0.708519      0.663475       0.749422
            job           blue-collar  2866.0     217.0 0.665870      0.637707       0.694174
            job          entrepreneur   443.0      31.0 0.70

## 4. Calibration

In [7]:
with stage("calibration"):
    cal = audit.calibration()
    print(cal[cal.subgroup != "GAP"][["attribute", "subgroup", "n", "calibration_intercept",
                                       "calibration_slope", "brier", "ece", "ici"]].to_string(index=False))

      attribute              subgroup       n  calibration_intercept  calibration_slope    brier      ece      ici
      age_group               Overall 13564.0              -0.000753           1.035305 0.086124 0.008412 0.007632
      age_group                 18-29  1503.0               0.173871           0.919426 0.117594 0.023295 0.022096
      age_group                 30-39  5463.0              -0.073593           1.022548 0.080216 0.009167 0.009343
      age_group                 40-49  3457.0              -0.104096           1.060448 0.069353 0.014614 0.013286
      age_group                 50-59  2615.0              -0.056130           1.145363 0.078555 0.015975 0.012775
      age_group                   60+   526.0               0.522790           0.659061 0.205427 0.087854 0.087940
            job               Overall 13564.0              -0.000753           1.035305 0.086124 0.008412 0.007632
            job                admin.  1509.0              -0.065812           0

## 5. Calibration curves

In [8]:
with stage("calibration_curves"):
    curves = audit.calibration_curves()
    print(curves.shape)

(8200, 7)
[calibration_curves] 53.99 s, 0 warning(s)


## 6. ROC curves

In [9]:
with stage("roc_curves"):
    roc = audit.roc_curves()
    print(roc.shape)

(8200, 7)
[roc_curves] 3.48 s, 0 warning(s)


## 7. Decision curve

In [10]:
with stage("decision_curve"):
    dc = audit.decision_curve()
    print(dc[dc.subgroup == "GAP"][["attribute", "gap_value", "gap_ci_low", "gap_ci_high"]].to_string(index=False))

      attribute  gap_value  gap_ci_low  gap_ci_high
      age_group   0.274165    0.218444     0.317074
            job   0.182547         NaN          NaN
        marital   0.041798    0.032993     0.059058
age_group × job   0.629630         NaN          NaN
[decision_curve] 8.65 s, 0 warning(s)


## 8. Estimability

In [11]:
with stage("estimability"):
    est = audit.estimability()
    print(f"{int(est.not_estimable_final.sum())} of {len(est)} cell-metrics not reportable")

428 of 930 cell-metrics not reportable
[estimability] 0.07 s, 0 warning(s)


## 9. Hosmer-Lemeshow

In [12]:
with stage("hosmer_lemeshow"):
    hl = audit.hosmer_lemeshow()
    print(hl[hl.subgroup != "GAP"][["attribute", "subgroup", "n", "hl_statistic", "hl_df",
                                     "hl_p_value", "n_bins_used"]].to_string(index=False))

      attribute              subgroup       n  hl_statistic  hl_df   hl_p_value  n_bins_used
      age_group               Overall 13564.0     13.713610   10.0 1.864600e-01         10.0
      age_group                 18-29  1503.0     14.130124   10.0 1.671386e-01         10.0
      age_group                 30-39  5463.0     13.195483   10.0 2.129466e-01         10.0
      age_group                 40-49  3457.0     15.232271   10.0 1.238203e-01         10.0
      age_group                 50-59  2615.0     12.014546   10.0 2.840842e-01         10.0
      age_group                   60+   526.0     47.493381   10.0 7.678425e-07         10.0
            job               Overall 13564.0     13.713610   10.0 1.864600e-01         10.0
            job                admin.  1509.0     14.719803   10.0 1.426162e-01         10.0
            job           blue-collar  2866.0     13.792906   10.0 1.826491e-01         10.0
            job          entrepreneur   443.0      5.870332   10.0 8.2

## 10. Cell diagnostics for one intersection

In [13]:
with stage("cell_diagnostics"):
    if INTERSECTION:
        cells = audit.cell_diagnostics()
        print(cells.to_string(index=False))
    else:
        print("single attribute; no intersection to diagnose")

      attribute              subgroup    n  n_events  shrunk  marginal_rate  cell_rate  shrinkage_factor
age_group × job        18-29 / admin.  218        26    True       0.169661   0.119266          0.507706
age_group × job   18-29 / blue-collar  299        35    True       0.169661   0.117057          0.429199
age_group × job  18-29 / entrepreneur   27         3    True       0.169661   0.111111          0.892783
age_group × job     18-29 / housemaid   14         4    True       0.169661   0.285714          0.941380
age_group × job    18-29 / management  226        50   False       0.169661   0.221239          0.498697
age_group × job       18-29 / retired    1         0    True       0.169661   0.000000          0.995572
age_group × job 18-29 / self-employed   61        15    True       0.169661   0.245902          0.786583
age_group × job      18-29 / services  175        16    True       0.169661   0.091429          0.562309
age_group × job       18-29 / student  201        51   

## 11. Recalibration ladder on the attribute with the largest intercept gap

In [14]:
gaps = cal[(cal.subgroup == "GAP") & (cal.gap_metric == "max_intercept_gap")
           & cal.attribute.isin(ATTRS)]
RANKED = [str(a) for a in gaps.sort_values("gap_value", ascending=False).attribute]


def zero_event_levels(attr):
    ev = pd.Series(y).groupby(marginals[attr]).sum()
    return [str(k) for k, v in ev.items() if v == 0]


LADDER_BY = RANKED[0]
ZERO = zero_event_levels(LADDER_BY)
print("largest intercept gap:", LADDER_BY, "| zero-event levels:", ZERO or "none")
# A level without events cannot be recalibrated; isitfair refuses (by design).
with stage("recalibration_ladder", expect_error=bool(ZERO)):
    ladder = audit.recalibration_ladder(by=LADDER_BY)
    print(ladder.to_string(index=False)[:3000])

largest intercept gap: job | zero-event levels: none


          rung attribute      subgroup       n  n_events  calibration_intercept  intercept_ci_low  intercept_ci_high  calibration_slope  slope_ci_low  slope_ci_high    brier  brier_ci_low  brier_ci_high      ece      ici  net_benefit        gap_metric  gap_value  event_floor below_floor not_estimable_final
          none       job       Overall 13564.0    1587.0              -0.000753         -0.051119           0.048297           1.035305      0.982245       1.102727 0.086124      0.082689       0.088925 0.008412 0.007632     0.032513               NaN        NaN         25.0       False               False
          none       job        admin.  1509.0     179.0              -0.065812         -0.212630           0.106603           0.861093      0.702239       1.035090 0.091353      0.082602       0.105974 0.023547 0.017991     0.030815               NaN        NaN         25.0       False               False
          none       job   blue-collar  2866.0     217.0               0.042

In [15]:
FALLBACK = next((a for a in RANKED[1:] if not zero_event_levels(a)), None) if ZERO else None
if FALLBACK:
    with stage("recalibration_ladder_fallback"):
        ladder = audit.recalibration_ladder(by=FALLBACK)
        print("by", FALLBACK)
        print(ladder.to_string(index=False)[:3000])
else:
    print("no fallback needed" if not ZERO else "no attribute without zero-event levels")

no fallback needed


## 12. Mitigation on a held-out calibration split

support2 uses the committed calibration partition. Every other dataset splits
its audited test half 50/50 (stratified on the outcome, `random_state=42`):
the calibrators are fitted on one half and the before/after audit runs on the
other, so no row is scored by a calibrator fitted on it.

In [16]:
with stage("mitigation_split"):
    if NAME == "support2":
        cal_df = pd.read_csv(registry.PRED / "support2_calib.csv", dtype={a: object for a in ATTRS})
        eval_df = df
    else:
        from sklearn.model_selection import train_test_split
        i_cal, i_eval = train_test_split(np.arange(len(df)), test_size=0.5, random_state=SEED,
                                         stratify=df["y_true"])
        cal_df, eval_df = df.iloc[i_cal].reset_index(drop=True), df.iloc[i_eval].reset_index(drop=True)
    for a in ATTRS:
        cal_df[a] = cal_df[a].fillna("Unknown")
    calib_data = (cal_df["y_true"].to_numpy(float), cal_df["y_proba_lr"].to_numpy(float),
                  {a: cal_df[a].to_numpy(object) for a in ATTRS})
    sf_eval = {a: eval_df[a].to_numpy(object) for a in ATTRS}
    if INTERSECTION:
        sf_eval[INTERSECTION] = None
    audit_eval = audit if NAME == "support2" else FairnessAudit(
        y_true=eval_df["y_true"].to_numpy(float), y_proba=eval_df["y_proba_lr"].to_numpy(float),
        sensitive_features=sf_eval, threshold=THRESHOLD, n_bootstrap=B, random_state=SEED, n_jobs=NJOBS)
    print(f"calibration n={len(cal_df)}, evaluation n={len(eval_df)}")

calibration n=6782, evaluation n=6782
[mitigation_split] 0.03 s, 0 warning(s)


In [17]:
with stage("mitigate_group_recalibration"):
    mitigated_gr = audit_eval.mitigate(method=GroupRecalibration(method="isotonic"),
                                       calibration_data=calib_data)
    a0 = ATTRS[0]
    keep = ["subgroup", "calibration_intercept", "calibration_slope", "brier", "ece"]
    before, after = audit_eval.calibration(), mitigated_gr[a0].calibration()
    print("BEFORE", a0); print(before[before.attribute == a0][keep].to_string(index=False))
    print("AFTER (group isotonic)"); print(after[after.attribute == a0][keep].to_string(index=False))

BEFORE age_group
subgroup  calibration_intercept  calibration_slope    brier      ece
 Overall               0.009466           1.033763 0.086231 0.010016
   18-29               0.198120           1.063079 0.113279 0.033988
   30-39              -0.126326           0.990013 0.078058 0.014403
   40-49              -0.047844           1.077855 0.071495 0.016559
   50-59               0.007334           1.120574 0.080847 0.021983
     60+               0.538648           0.587290 0.212510 0.115005
     GAP                    NaN                NaN      NaN      NaN
     GAP                    NaN                NaN      NaN      NaN
AFTER (group isotonic)
subgroup  calibration_intercept  calibration_slope    brier      ece
 Overall               0.024577           0.711248 0.086202 0.015942
   18-29               0.036912           1.141860 0.112975 0.041195
   30-39              -0.114408           0.740188 0.078888 0.017004
   40-49               0.140753           0.583192 0.071130 0.0

In [18]:
with stage("mitigate_wasserstein"):
    mitigated_w = audit_eval.mitigate(method=WassersteinPostprocessing(random_state=SEED),
                                      calibration_data=calib_data)
    d_after = mitigated_w[a0].discrimination()
    print(d_after[(d_after.attribute == a0) & (d_after.subgroup == "GAP")][
        ["gap_metric", "gap_value", "gap_ci_low", "gap_ci_high"]].to_string(index=False))

                  gap_metric  gap_value  gap_ci_low  gap_ci_high
equal_opportunity_difference   0.179683    0.131491     0.255529
         predictive_equality   0.056134    0.036988     0.082529
          equalized_odds_gap   0.179683    0.131491     0.255529
          statistical_parity   0.020607    0.015870     0.053842
[mitigate_wasserstein] 3.19 s, 0 warning(s)


## 13. Report: HTML, then PDF

In [19]:
REPORT = REPORTS / f"{NAME}_B{B}.html"
with stage("report_html"):
    audit_eval.report(out=REPORT, title=f"{NAME}: hardening audit (B={B})",
                      metadata={"Dataset": NAME, "Model": "LogisticRegression (registry)",
                                "Threshold": str(THRESHOLD), "Bootstrap": f"B={B}, random_state={SEED}"},
                      mitigation_audits=mitigated_gr)
    print(REPORT, REPORT.stat().st_size, "bytes")

C:\Users\hugoa\Desktop\isitfair\hardening\reports\bank_B50.html 4449616 bytes
[report_html] 109.82 s, 3 warning(s)


In [20]:
with stage("report_pdf"):
    pdf_path = report_to_pdf(REPORT)
    print(pdf_path, pdf_path.stat().st_size, "bytes")

C:\Users\hugoa\Desktop\isitfair\hardening\reports\bank_B50.pdf 1196728 bytes
[report_pdf] 43.12 s, 0 warning(s)


## 14. Input-type variants

The same audit (B = 20 to keep this cheap) with inputs passed in the forms
real pipelines produce. Each variant is classified as `raises`, `warns`,
`ok-identical` (results equal the plain-numpy baseline exactly), or
`ok-DIFFERENT` (runs silently but gives different numbers: a silent
misbehaviour unless the input itself changed, as in the ties variant).

In [21]:
VB = 20


def run_variant(name, y_in, p_in, sf_in, compare=True, ref=None):
    out = {"status": None, "warnings": [], "error": None}
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        try:
            a = FairnessAudit(y_true=y_in, y_proba=p_in, sensitive_features=sf_in,
                              threshold=THRESHOLD, n_bootstrap=VB, random_state=SEED, n_jobs=NJOBS)
            frames = {"discrimination": a.discrimination(), "calibration": a.calibration(),
                      "hosmer_lemeshow": a.hosmer_lemeshow(), "decision_curve": a.decision_curve()}
        except Exception as exc:
            out["error"] = f"{type(exc).__name__}: {exc}"
            frames = None
    out["warnings"] = sorted({f"{w.category.__name__}: {str(w.message)[:200]}" for w in caught})
    if frames is None:
        out["status"] = "raises"
    else:
        num_bad = {k: int(np.isinf(f.select_dtypes("number").to_numpy(float)).sum()) for k, f in frames.items()}
        out["inf_values"] = num_bad
        if ref is not None and compare:
            same = all(frames[k].equals(ref[k]) for k in frames)
            out["identical_to_baseline"] = same
            out["status"] = ("warns, " if out["warnings"] else "") + ("ok-identical" if same else "ok-DIFFERENT")
            if not same:
                diffs = {}
                for k in frames:
                    fa, fb = frames[k], ref[k]
                    if fa.shape == fb.shape:
                        na = fa.select_dtypes("number").to_numpy(float)
                        nb = fb.select_dtypes("number").to_numpy(float)
                        diffs[k] = float(np.nanmax(np.abs(na - nb))) if na.size else 0.0
                    else:
                        diffs[k] = f"shape {fa.shape} vs {fb.shape}"
                out["max_abs_diff"] = diffs
        else:
            out["status"] = ("warns, " if out["warnings"] else "") + "ok"
    RECORD["variants"][name] = out
    print(f"{name:<28} {out['status']}" + (f"  {out['error']}" if out["error"] else ""))
    for w in out["warnings"][:5]:
        print("     ", w)
    (OUT / f"{NAME}_B{B}.json").write_text(json.dumps(RECORD, indent=2, default=str))
    return frames


with stage("variants"):
    base = run_variant("baseline_numpy", y, p, sf, compare=False)
    s_attr = {a: pd.Series(v, dtype="category") for a, v in marginals.items()}
    run_variant("category_attributes", y, p, {**s_attr, **({INTERSECTION: None} if INTERSECTION else {})}, ref=base)
    run_variant("Int64_outcome", pd.Series(y.astype(int), dtype="Int64"), p, sf, ref=base)
    run_variant("bool_outcome", y.astype(bool), p, sf, ref=base)
    run_variant("bool_outcome_series", pd.Series(y.astype(bool)), p, sf, ref=base)
    perm = np.random.default_rng(SEED).permutation(len(y))
    idx = pd.Index(perm + 10_000)
    run_variant("shuffled_index_consistent",
                pd.Series(y, index=idx), pd.Series(p, index=idx),
                {**{a: pd.Series(v, index=idx) for a, v in marginals.items()},
                 **({INTERSECTION: None} if INTERSECTION else {})}, ref=base)
    # y sorted/shuffled with its labels, p left in the original order: the
    # classic silent misalignment, which isitfair is documented to reject.
    run_variant("shuffled_index_misaligned",
                pd.Series(y).iloc[perm], pd.Series(p), sf, ref=base)
    p_ties = np.round(p, 2)
    k = max(2, len(p) // 50)  # ~2% of rows exactly 0 or 1, never all of them
    pos = np.random.default_rng(SEED).choice(len(p), size=k, replace=False)
    p_ties[pos[: k // 2]] = 0.0
    p_ties[pos[k // 2 :]] = 1.0
    run_variant("ties_and_exact_0_1", y, p_ties, sf, compare=False)

baseline_numpy               ok


category_attributes          ok-identical


Int64_outcome                ok-identical


bool_outcome                 ok-identical


bool_outcome_series          ok-identical


shuffled_index_consistent    ok-identical
shuffled_index_misaligned    raises  ValueError: `y_proba` and `y_true` are pandas objects with the same index labels in a different order. FairnessAudit aligns inputs by position, not by index, so the rows would be mismatched. Reorder them consistently, e.g. `s.loc[other.index]`, or pass `.to_numpy()` arrays built from one DataFrame.


ties_and_exact_0_1           ok
[variants] 321.16 s, 0 warning(s)


## 15. Summary

In [22]:
unexpected = {k: v["error"] for k, v in RECORD["stages"].items()
              if v["error"] and not v["expected_error"]}
RECORD["unexpected_errors"] = unexpected
RECORD["total_seconds"] = round(sum(v["seconds"] or 0 for v in RECORD["stages"].values()), 2)
(OUT / f"{NAME}_B{B}.json").write_text(json.dumps(RECORD, indent=2, default=str))
print(pd.DataFrame({k: {"seconds": v["seconds"], "n_warnings": len(v["warnings"]),
                        "error": v["error"]} for k, v in RECORD["stages"].items()}).T.to_string())
print("unexpected errors:", unexpected or "none")
if STRICT:
    assert not unexpected, unexpected

                              seconds  n_warnings  error
load_validate                   0.020         0.0    NaN
construct                       0.064         0.0    NaN
discrimination                 13.382         0.0    NaN
calibration                    25.268         0.0    NaN
calibration_curves             53.993         0.0    NaN
roc_curves                      3.485         0.0    NaN
decision_curve                  8.653         0.0    NaN
estimability                    0.066         0.0    NaN
hosmer_lemeshow                21.034         0.0    NaN
cell_diagnostics                0.042         0.0    NaN
recalibration_ladder           19.816         0.0    NaN
mitigation_split                0.032         0.0    NaN
mitigate_group_recalibration   13.764         0.0    NaN
mitigate_wasserstein            3.189         0.0    NaN
report_html                   109.824         1.0    NaN
report_pdf                     43.115         0.0    NaN
variants                      3